# Clean the Australia service list and insert it into DB

This notebook builds a simple data pipeline for the uploaded Excel file and writes a cleaned CSV.

## Cleaning rules
- read the worksheet using the real header row
- drop completely empty rows and columns
- trim and normalize whitespace in text fields
- convert blank strings to missing values
- rename the columns for psql
- keep postcodes as 4-digit text values
- treat blank place counts as `0` for:
  - `residential_places`
  - `home_care_places`
  - `restorative_care_places`
- round latitude/longitude to 6 decimals
- round funding to 2 decimals
- remove exact duplicate rows


In [ ]:
from pathlib import Path
import re
import pandas as pd

INPUT_XLSX = Path("../input/Service-List-2025-Australia_300126.xlsx")

INPUT_XLSX

(PosixPath('input/Service-List-2025-Australia_300126.xlsx'),
 PosixPath('output/service_list_2025_australia_clean.csv'))

In [4]:
df = pd.read_excel(INPUT_XLSX, sheet_name=0, header=2)
df.columns

Index(['Service Name', 'Physical Address', 'Physical Suburb', 'Physical State',
       'Physical Post Code', '2018 Aged Care Planning Region (ACPR)',
       'Care Type', 'Residential Places', 'Home Care Places',
       'Restorative Care Places', 'Provider Name', 'Organisation Type',
       'ABS Remoteness', '2019 MMM Code', '2016 SA2 Code', '2016 SA2 Name',
       '2016 SA3 Code', '2016 SA3 Name', '2023 LGA Name', '2023 LGA Code',
       '2017 PHN Code', '2017 PHN Name', 'Latitude', 'Longitude',
       '2024-25 Australian Government Funding'],
      dtype='str')

In [ ]:
def clean_service_list(input_path: Path):
    # Read the Excel file, skipping the first two rows to get to the real header.
    df = pd.read_excel(input_path, sheet_name=0, header=2)

    # Define the new column names
    column_mapping = {
        'Service Name': 'service_name',
        'Physical Address': 'physical_address',
        'Physical Suburb': 'physical_suburb',
        'Physical State': 'physical_state',
        'Physical Post Code': 'physical_post_code',
        '2018 Aged Care Planning Region (ACPR)': 'aged_care_planning_region_acpr_2018',
        'Care Type': 'care_type',
        'Residential Places': 'residential_places',
        'Home Care Places': 'home_care_places',
        'Restorative Care Places': 'restorative_care_places',
        'Provider Name': 'provider_name',
        'Organisation Type': 'organisation_type',
        'ABS Remoteness': 'abs_remoteness',
        '2019 MMM Code': 'mmm_code_2019',
        '2016 SA2 Code': 'sa2_code_2016',
        '2016 SA2 Name': 'sa2_name_2016',
        '2016 SA3 Code': 'sa3_code_2016',
        '2016 SA3 Name': 'sa3_name_2016',
        '2023 LGA Name': 'lga_name_2023',
        '2023 LGA Code': 'lga_code_2023',
        '2017 PHN Code': 'phn_code_2017',
        '2017 PHN Name': 'phn_name_2017',
        'Latitude': 'latitude',
        'Longitude': 'longitude',
        '2024-25 Australian Government Funding': 'australian_government_funding_2024_25'
    }
    df = df.rename(columns=column_mapping)

    # Drop completely empty rows and columns.
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    # Clean up all text columns: normalize whitespace and convert blanks to missing values.
    object_cols = df.select_dtypes(include="object").columns
    for col in object_cols:
        s = df[col].astype("string")
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
        s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df[col] = s

    # Format postcodes as 4-digit strings, preserving leading zeros.
    if "physical_post_code" in df.columns:
        df["physical_post_code"] = (
            pd.to_numeric(df["physical_post_code"], errors="coerce")
            .round()
            .astype("Int64")
            .astype("string")
            .str.zfill(4)
        )

    # Convert blank or non-numeric place counts to 0.
    for col in ["residential_places", "home_care_places", "restorative_care_places"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("Int64")    
    
    # Ensure identifier codes are treated as strings.
    for col in ["sa2_code_2016", "sa3_code_2016", "lga_code_2023"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64").astype("string")

    # Convert MMM code to a number.
    if "mmm_code_2019" in df.columns:
        df["mmm_code_2019"] = pd.to_numeric(df["mmm_code_2019"], errors="coerce").astype("Int64")

    # Round geographic coordinates to 6 decimal places.
    for col in ["latitude", "longitude"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").round(6)

    # Round funding to 2 decimal places for currency.
    if "australian_government_funding_2024_25" in df.columns:
        df["australian_government_funding_2024_25"] = (
            pd.to_numeric(df["australian_government_funding_2024_25"], errors="coerce").round(2)
        )

    # Remove Home care
    df = df[df['care_type'] != 'Home Care'].copy()
    # Remove duplicate rows and count how many were removed.
    rows_before = len(df)
    dedup_keys = ['service_name', 'care_type', 'latitude', 'longitude']
    before = len(df)
    df = (
        df.sort_values('residential_places', ascending=False, na_position='last')
        .drop_duplicates(subset=dedup_keys, keep='first')
    )
    print(f"Collapsed {before - len(df)} same-facility duplicates")

    df = df.drop_duplicates().reset_index(drop=True)
    duplicates_removed = rows_before - len(df)

    # Return a summary of the cleaning process.
    summary = {
        "input_rows": rows_before,
        "output_rows": len(df),
        "duplicates_removed": duplicates_removed,
        "column_count": df.shape[1],        
    }
    return df, summary

NameError: name 'Path' is not defined

In [6]:
clean_df, summary = clean_service_list(INPUT_XLSX)
summary

/var/folders/w4/l4h21lrj0d59d23sfzvj3t5m0000gn/T/ipykernel_80367/4101083163.py:39: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include="object").columns


{'input_rows': 5378,
 'output_rows': 5370,
 'duplicates_removed': 8,
 'column_count': 25}

In [7]:
clean_df.head()

,service_name,physical_address,physical_suburb,physical_state,physical_post_code,aged_care_planning_region_acpr_2018,care_type,residential_places,home_care_places,restorative_care_places,...,sa2_name_2016,sa3_code_2016,sa3_name_2016,lga_name_2023,lga_code_2023,phn_code_2017,phn_name_2017,latitude,longitude,australian_government_funding_2024_25
0,Braidwood Multi-Purpose Service,77 Monkittee Street,BRAIDWOOD,NSW,2622,Southern Highlands,Multi-Purpose Service,37,2,0,...,Braidwood,10102,Queanbeyan,Queanbeyan-Palerang,16490,PHN106,South Eastern NSW,-35.442764,149.805407,2683564.28
1,Dellacourt,42 NICHOLSON Place,WEST ALBURY,NSW,2640,Riverina/Murray,Residential,122,0,0,...,Albury - South,10901,Albury,Albury,10050,PHN205,Murray,-36.075401,146.890896,13628406.71
2,Brerrina Multi-Purpose Service,56 DOYLE Street,BRERRINA,NSW,2839,Orana Far West,Multi-Purpose Service,12,4,0,...,Bourke - Brewarrina,10501,Bourke - Cobar - Coonamble,Brewarrina,11200,PHN107,Western NSW,-29.961591,146.864313,1372056.96
3,BaptistCare Maranoa Centre - Alstonville,15 The Avenue -,ALSTONVILLE,NSW,2477,Far North Coast,Residential,90,0,0,...,Ballina Region,11201,Richmond Valley - Coastal,Ballina,10250,PHN109,North Coast,-28.840536,153.436634,10456454.14
4,Urana Multi-Purpose Service,127-129 Princess Street,URANA,NSW,2645,Riverina/Murray,Multi-Purpose Service,19,0,0,...,Corowa Region,10903,Upper Murray exc. Albury,Federation,12870,PHN110,Murrumbidgee,-35.325391,146.267697,1591105.21


In [8]:
print(f"Blank rows remaining: {int(clean_df.isna().all(axis=1).sum())}")
print(f"Exact duplicates remaining: {int(clean_df.duplicated().sum())}")
print(f"Whitespace-only text cells remaining: {int(sum(clean_df[col].fillna('').str.fullmatch(r'\\s*').sum() for col in clean_df.select_dtypes(include='string').columns))}")

Blank rows remaining: 0
Exact duplicates remaining: 0
Whitespace-only text cells remaining: 0


# Insert Clean Data into DB

In [9]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/aged_care_services.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)

Schema created or already exists.
carelink


In [10]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
# Assume df already exists and is clean
df_insert = clean_df.copy()

target_columns = [
    "service_name",
    "physical_address",
    "physical_suburb",
    "physical_state",
    "physical_post_code",
    "aged_care_planning_region_acpr_2018",
    "care_type",
    "residential_places",
    "home_care_places",
    "restorative_care_places",
    "provider_name",
    "organisation_type",
    "abs_remoteness",
    "mmm_code_2019",
    "sa2_code_2016",
    "sa2_name_2016",
    "sa3_code_2016",
    "sa3_name_2016",
    "lga_name_2023",
    "lga_code_2023",
    "phn_code_2017",
    "phn_name_2017",
    "latitude",
    "longitude",
    "australian_government_funding_2024_25",
]

df_insert = df_insert[target_columns].copy()

# text columns
text_cols = [
    "service_name",
    "physical_address",
    "physical_suburb",
    "physical_state",
    "physical_post_code",
    "aged_care_planning_region_acpr_2018",
    "care_type",
    "provider_name",
    "organisation_type",
    "abs_remoteness",
    "sa2_code_2016",
    "sa2_name_2016",
    "sa3_code_2016",
    "sa3_name_2016",
    "lga_name_2023",
    "lga_code_2023",
    "phn_code_2017",
    "phn_name_2017",
]

for col in text_cols:
    df_insert[col] = df_insert[col].astype("string").str.strip()

# integer columns
int_cols = [
    "residential_places",
    "home_care_places",
    "restorative_care_places",
    "mmm_code_2019",
]

for col in int_cols:
    df_insert[col] = pd.to_numeric(df_insert[col], errors="coerce").astype("Int64")

# float columns
float_cols = [
    "latitude",
    "longitude",
    "australian_government_funding_2024_25",
]

for col in float_cols:
    df_insert[col] = pd.to_numeric(df_insert[col], errors="coerce")

In [11]:
# -----------------------------
# 4. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    "aged_care_services",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(df_insert)} rows into aged_care_services")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM aged_care_services"))
    row_count = result.scalar()

print("Rows in table:", row_count)

Inserted 5370 rows into aged_care_services
Rows in table: 5370
